In [1]:
EXPERIMENTAL_ID="product_search_custom_scoring"
START_DATE="2025-08-23"

In [2]:
import requests
import pandas as pd

url = "http://dic.summerfarm.net/synonym.txt"
keywords_to_include=requests.get(url).text.split("\n")
keywords_to_include=[keyword.strip().split(",") for keyword in keywords_to_include]

indivitual_keywords_to_include=[]
for arr in keywords_to_include:
    for keyword in arr:
        indivitual_keywords_to_include.append(keyword)

print(indivitual_keywords_to_include[:10])

indivitual_keywords_to_include_df=pd.DataFrame(indivitual_keywords_to_include,columns=["keyword_synonym"])
indivitual_keywords_to_include_df.head(5)

['葡萄', '巨峰', '巨峰葡萄', '芋泥', '芋头泥', '香芋泥', '芋茸', '罐头', '荔枝罐头', '西柚粒罐头']


,keyword_synonym
0,葡萄
1,巨峰
2,巨峰葡萄
3,芋泥
4,芋头泥


In [5]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

        #         ,CASE
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
        #     ELSE '中频搜索词'
        #  END

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,RANK() OVER (PARTITION BY "dontcarte" ORDER BY COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) DESC) as search_cnt_rnk
        ,"不区分频次" AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df['搜索频次标签']=top_query_df['search_cnt_rnk'].apply(lambda x: 'top400' if x <= 400 else 'top400以外')
top_query_df.head(20)

2025-08-25 15:07:28 - INFO - Tunnel session created: <InstanceDownloadSession id=20250825150728b436f60b17e598bb project_name=summerfarm_ds instance_id=20250825070655679gb6qpzbt0o2>
2025-08-25 15:07:30 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHE

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,search_cnt_rnk,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,芒果,12001,221049,73272,5.2,101.0,0.0,6.4,3.0,7.0,13.0,1,top400,0.331474,60,高点击率词
1,牛奶,11604,102319,39909,5.4,290.0,0.0,11.1,1.0,6.0,20.0,4,top400,0.390045,60,高点击率词
2,柠檬,10482,118408,39835,5.1,554.0,0.0,8.6,3.0,7.0,12.0,3,top400,0.336422,60,高点击率词
3,草莓,8744,119559,42252,3.0,392.0,0.0,6.2,1.0,4.0,7.0,2,top400,0.353399,60,高点击率词
4,安佳,8093,60801,19926,2.6,211.0,0.0,6.9,0.0,2.0,8.0,8,top400,0.327725,60,高点击率词
5,蓝莓,7577,90528,31207,1.9,154.0,0.0,3.9,1.0,2.0,5.0,6,top400,0.344722,60,高点击率词
6,奶油,7531,73326,18186,16.6,284.0,0.0,22.9,8.0,23.0,44.0,7,top400,0.248016,60,低点击率词
7,西瓜,6019,102216,31075,4.2,239.0,0.0,6.3,3.0,5.0,9.0,5,top400,0.304013,60,高点击率词
8,黄油,5678,41844,9392,11.8,194.0,0.0,14.7,6.0,20.0,29.0,12,top400,0.224453,60,低点击率词
9,荔枝,4959,51783,15069,2.2,212.0,0.0,7.3,1.0,2.0,5.0,9,top400,0.291003,60,高点击率词


In [6]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    """
    从SLS(Simple Log Service)获取指定日期的用户变体数据。

    Args:
        day (datetime): 要获取数据的日期。
        check_if_local_exist (bool): 是否检查本地数据库中是否已存在数据，默认为True。

    Returns:
        pd.DataFrame: 包含用户变体数据的DataFrame。
    """
    # 构建数据库文件名和表名
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    # 连接到SQLite数据库
    conn = sqlite3.connect(db_file_name)

    # 如果设置为检查本地数据
    if check_if_local_exist:
        try:
            # 尝试从数据库中读取数据
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            # 关闭数据库连接
            conn.close()
            # 返回读取的数据
            return df
        except pd.io.sql.DatabaseError:
            # 如果表不存在，则忽略错误
            pass

    # 构建SLS查询语句
    query = f"""
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{{digit}}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{{[^}}]+\}}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"{EXPERIMENTAL_ID}"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000
"""
    # 设置查询的起始时间和结束时间
    print(query)
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    # 从SLS获取数据
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",  # 指定SLS项目
        logstore="xm-mall",  # 指定SLS日志库
        from_time=from_time,  # 指定查询起始时间
        to_time=to_time,  # 指定查询结束时间
    )

    # 将search_times列中的缺失值填充为1，并转换为整数类型
    _df["search_times"] = _df["search_times"].fillna(1).astype(int)
    # 将variant_list列中的缺失值填充为"none"
    _df["variant_list"] = _df["variant_list"].fillna("none")

    # 如果DataFrame不为空
    if not _df.empty:
        # 删除不需要的列
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        # 将数据写入SQLite数据库，如果表已存在则替换
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    # 关闭数据库连接
    conn.close()
    # 返回数据
    return _df


# 创建一个空的DataFrame来存储所有日期的用户变体数据
all_user_variant_df = pd.DataFrame()
# 设置起始日期和结束日期
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
end_date = datetime.now()
# 从起始日期开始循环，直到结束日期
current_date = start_date
while current_date <= end_date:
    # 检查是否是今天
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    # 如果是今天,则跳过，因为今天的数据可能不完整
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    # 获取当前日期的用户变体数据
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    # 将当前日期的数据添加到总的DataFrame中
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    # 日期增加一天
    current_date += timedelta(days=1)

# 显示前10行数据
all_user_variant_df.head(10)

今天的数据还未完整，跳过:2025-08-25 00:00:00


,api,page_name,experiment_id,type,uid,ds,search_times,variant_list
0,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,464561,20250823,17,V1
1,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,315101,20250823,14,V1
2,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,314047,20250823,2,V4
3,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,450340,20250823,2,V2
4,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,442157,20250823,7,V1
5,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,543499,20250823,1,V3
6,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,449005,20250823,3,V2
7,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,384996,20250823,1,V2
8,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,406426,20250823,1,V4
9,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,139223,20250823,4,V2


In [7]:
import pandasql
stats=pandasql.sqldf("""select ds,variant_list,count(distinct uid) unique_user 
                     from all_user_variant_df group by ds,variant_list order by ds desc,variant_list""")

display(stats)

,ds,variant_list,unique_user
0,20250824,V1,1815
1,20250824,V2,1891
2,20250824,V3,1861
3,20250824,V4,1827
4,20250823,V1,1954
5,20250823,V2,1991
6,20250823,V3,1944
7,20250823,V4,2035


In [8]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

今天的数据还未完整，跳过:2025-08-25 00:00:00


In [9]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

今天的数据还未完整，跳过:2025-08-25 00:00:00


,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,null,加入购物车,N001S01R005,加购弹窗,56,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:2772",20250823,安佳,cl,580,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:2772","name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no"
1,null,加入购物车,5476680107,加购弹窗,1469,"name:加入购物车,pid:加购弹窗,sku:5476680107,pdid:1469,stock:20",20250823,蜜瓜,cl,434097,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:5476680107,pdid:1469,stock:20","name:Search,word:PET在线定制速戳,linkShadingWord:[object Object],isTiming:no"
2,1,香草加州阳光桶冰淇淋,836172146601,goods,4902,"idx:1,name:香草加州阳光桶冰淇淋,pid:goods,sku:836172146601,salePrice:68,pdid:4902,stock:10000,ext:cross",20250823,冰淇淋,cl,516096,/search/goods-new,"idx:1,name:香草加州阳光桶冰淇淋,pid:goods,sku:836172146601,salePrice:68,pdid:4902,stock:10000,ext:cross","name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
3,0,雀巢淡奶油 1L*12盒,N001Q01C001,唤起购买,176,undefined,20250823,雀巢,cl,580,/search/goods-new,undefined,"name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no"
4,null,加入购物车,647471767,加购弹窗,377,"name:加入购物车,pid:加购弹窗,sku:647471767,pdid:377,stock:253",20250823,越南红心火龙果,cl,499641,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:647471767,pdid:377,stock:253","name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no"
5,null,加入购物车,N001Q01C001,加购弹窗,176,"name:加入购物车,pid:加购弹窗,sku:N001Q01C001,pdid:176,stock:328",20250823,雀巢,cl,580,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:N001Q01C001,pdid:176,stock:328","name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no"
6,4,龙田牌白砂糖 50KG*1包/50kg(破袋特惠，售出不退不换~),791580760417,唤起购买,8476,undefined,20250823,白砂糖,cl,564001,/search/goods-new,undefined,"name:Search,word:PET在线定制速戳,linkShadingWord:[object Object],isTiming:no"
7,3,蒙特瑞草莓 300g*1盒/一级/4*5（亮盒）,611184245,唤起购买,452,undefined,20250823,草莓,cl,123069,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"
8,1,晴王青提(阳光玫瑰) 净重8-9斤/普通/标准规格(存在轻微掉粒、压伤、泛黄情况),16823457730,goods,1640,"idx:1,name:晴王青提(阳光玫瑰) 净重8-9斤/普通/标准规格(存在轻微掉粒、压伤、泛黄情况),pid:goods,sku:16823457730,salePrice:48,pdid...",20250823,青提,cl,434097,/search/goods-new,"idx:1,name:晴王青提(阳光玫瑰) 净重8-9斤/普通/标准规格(存在轻微掉粒、压伤、泛黄情况),pid:goods,sku:16823457730,salePrice:48,pdid...","name:Search,word:PET在线定制速戳,linkShadingWord:[object Object],isTiming:no"
9,2,湖北夏橙 5斤*1包/一级/标准规格,5477727035,唤起购买,1003,undefined,20250823,橙子,cl,473016,/search/goods-new,undefined,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no"


In [10]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    search_query=_dict["search_query"]
    if not search_query or f"{search_query}" == "":
        search_query = _dict["linkInfo"]
        # 搜索pdName，如果没找到，则search_query为空字符串
        match = re.search(r'pdName:([^,]+)', search_query)
        search_query = match.group(1) if match else ""
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        sku_info["search_query"] = search_query
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:2772",N001S01R005,加入购物车,None,加购弹窗,56,"name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no",安佳
1,"name:加入购物车,pid:加购弹窗,sku:5476680107,pdid:1469,stock:20",5476680107,加入购物车,None,加购弹窗,1469,"name:Search,word:PET在线定制速戳,linkShadingWord:[object Object],isTiming:no",蜜瓜
2,"idx:1,name:香草加州阳光桶冰淇淋,pid:goods,sku:836172146601,salePrice:68,pdid:4902,stock:10000,ext:cross",836172146601,香草加州阳光桶冰淇淋,1,goods,4902,"name:Search,word:暑期爆款配方,linkShadingWord:[object Object],isTiming:no",冰淇淋
3,undefined,N001Q01C001,雀巢淡奶油 1L*12盒,0,唤起购买,176,"name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no",雀巢
4,"name:加入购物车,pid:加购弹窗,sku:647471767,pdid:377,stock:253",647471767,加入购物车,None,加购弹窗,377,"name:Search,word:海苔酥脆松,linkShadingWord:[object Object],isTiming:no",越南红心火龙果


In [11]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_name', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [12]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
5,加购弹窗,28993
6,唤起购买,24653
2,goods,13426
8,横版筛选栏,784
7,商品列表,648
3,mini榜单,443
9,竖版筛选栏,41
0,AI采购,15
10,鲜沐农场AI使用协议,3
1,AI问题,1


In [13]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [14]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,100041,V1,20250823,top400,1,0,0,1,1,5.0,5,5,1,1
1,100041,V1,20250824,top400,1,0,0,1,1,0.0,0,0,1,1
2,100041,V1,20250824,top400以外,1,0,0,1,0,11.0,11,11,1,1
3,10008,V2,20250824,top400,1,0,0,1,1,0.0,0,0,1,1
4,100111,V3,20250823,top400,1,0,0,1,1,3.0,3,3,1,1


In [15]:
null_search_query_df = pandasql.sqldf(
    """select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                    count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [16]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
inner join indivitual_keywords_to_include_df c on a.search_query = c.keyword_synonym
left join top_query_df b on a.search_query = b.query
where c.keyword_synonym is not null
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)

print("unique sample_query:", user_view_with_variant_statistics_df['sample_query'].unique())

unique sample_query: ['芒果' '草莓' '西瓜' '淡奶油' '奶酪' '高筋面粉' '椰浆' '水蜜桃' '苹果' '水果' '葡萄' '椰乳' '牛奶' '橙子'
 '芋圆' '脆啵啵' '白巧克力' '巧克力' '凤梨' '牛油果' '梨' '澄善HPP速冻凤梨颗粒浆' '蓝莓' '红柚' '奇异果'
 '马斯卡彭' '青提' '鲜牛奶' '芋泥' '低筋粉' '芭乐' '果汁' '砀山梨' '荔枝' '脆波波' '苏打水' '稀奶油' '气泡水'
 '鲜奶' '布丁' '巨峰' '西柚' '提子' '桃子' '小芋圆' '果糖' '冰淇淋' '冰激凌' '果酱' '菠萝' '红凯特芒'
 '咖啡杯' '黑巧' '番茄' '阳光玫瑰' '白巧' '圣女果' '夏黑' '晶球' '糖浆' '艾恩' '桃' '巨峰葡萄' '酒'
 '葡萄罐头' 'PET冷饮杯' '猕猴桃' '葡萄汁' '杯子' '低筋面粉' '夏黑葡萄' '湖北夏橙' '青凯特芒' '原味' '燕麦奶'
 '荔枝罐头' '波波晶球' '杯' '台农芒果' '水仙芒' '海南水仙芒' '番石榴' '丸子' '金煌芒' '大芋圆' '伦晚橙' '鱼胶'
 '烤肠' '草莓果酱' '澳洲脐橙' '皇冠梨' '樱桃' '大青芒' '雪克杯' '红富士' '盖子' '贡梨' '鲜牛乳' '干酪' '香肠'
 '椰奶' '红颜草莓' '台农' '夏橙' '红西柚' '橘子' '罐头' '水牛奶' '薯条' '四川金煌芒' '艾恩摩尔' '高筋粉'
 '车厘子' '南非橙' '柚子' '红提' '澄善HPP速冻葡萄汁' '黑巧克力' '埃及橙' '晴王青提' '冷饮杯' '贵妃芒' '土豆'
 '红薯' '杯盖' '水牛乳' '金果' '果浆' '蜜薯' '西柚粒罐头' '小台农']


In [17]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)
# 定义计数列名列表
count_columns = ["商品详情cnt", "加入购物车cnt", "唤起购买cnt", "首屏总点击cnt", "总点击cnt"]
# 遍历计数列
for col in count_columns:
    # 将空值填充为0并转换为整数类型
    all_data_df[col] = all_data_df[col].fillna(0).astype(int)

# 创建 "用户是否点击" 列
all_data_df["用户是否点击"] = (all_data_df["总点击cnt"] > 0).astype(int)

# 定义费率计算相关列名列表
rate_columns = [
    ("sku_click_rate", "商品详情cnt", "商品查看cnt"), # 商品详情点击率
    ("add_cart_rate", "加入购物车cnt", "商品查看cnt"), # 加入购物车率
    ("popup_click_rate", "唤起购买cnt", "商品查看cnt"), # 唤起购买率
]

# 遍历费率列
for rate_col, num_col, den_col in rate_columns:
    # 计算费率，空值填充0，保留5位小数，转换为浮点数
    all_data_df[rate_col] = (all_data_df[num_col] / all_data_df[den_col]).fillna(0).round(5).astype(float)

all_data_df.head(5)

,uid,ds,variant_list,搜索频次标签,sample_query,商品查看cnt,查看SKU_cnt,查看搜索词cnt,max查看位置,搜索翻页数cnt,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt,用户是否点击,sku_click_rate,add_cart_rate,popup_click_rate
0,,20250824,None,top400,芒果,24,7,1,2,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.00000,0.00000,0.00000
1,100041,20250823,V1,top400,草莓,24,7,1,6,8.0,1,0,0,1,1,5.0,5.0,5.0,1.0,1.0,1,0.04167,0.00000,0.00000
2,100111,20250823,V3,top400,西瓜,5,5,1,4,1.0,1,0,0,1,1,3.0,3.0,3.0,1.0,1.0,1,0.20000,0.00000,0.00000
3,100171,20250823,V3,top400,芒果,26,15,2,10,5.0,0,2,4,4,2,7.8,15.0,2.0,5.0,4.0,1,0.00000,0.07692,0.15385
4,100530,20250824,V4,top400,淡奶油,24,24,2,13,2.0,1,0,0,1,1,3.0,3.0,3.0,1.0,1.0,1,0.04167,0.00000,0.00000


In [18]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [19]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        # print(
        #     f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        # )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            # print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [20]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "唤起购买cnt",
    "总点击cnt",
    "商品详情cnt",
    "加入购物车cnt",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            # print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


title_all = f"搜索AB--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
all_p_values_df

写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--avg点击位置_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--唤起购买cnt_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--总点击cnt_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--商品详情cnt_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--加入购物车cnt_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--商品查看cnt_p-value分布-0823~0824.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0823~0824.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric,搜索频次
0,V1,0.9519,0.0624,0.1655,-0.50,0.00,0.055560,0.171430,0.324560,0.333330,0.774999,1.000000,3.33333,55,886,354,0823~0824,sku_click_rate,top400
1,V2,1.0000,0.0627,0.1445,0.00,0.00,0.066670,0.198048,0.333330,0.400000,0.666670,0.854997,2.66667,57,902,382,0823~0824,sku_click_rate,top400
2,V3,0.2369,0.0573,0.1252,-8.60,0.00,0.058820,0.166670,0.285710,0.333330,0.600000,0.771429,1.33333,49,852,362,0823~0824,sku_click_rate,top400
3,V4,0.3272,0.0679,0.1696,8.20,0.00,0.070710,0.200000,0.333330,0.400000,0.750000,1.000000,2.00000,61,898,389,0823~0824,sku_click_rate,top400
4,V1,0.1114,0.0857,0.1853,-60.22,0.00,0.097220,0.250000,0.339997,0.512666,0.903331,0.951666,1.00000,3,30,10,0823~0824,sku_click_rate,top400以外
5,V2,1.0000,0.2155,0.6049,0.00,0.00,0.191668,0.370833,0.658336,1.255000,3.085000,3.542500,4.00000,7,31,14,0823~0824,sku_click_rate,top400以外
6,V3,0.4909,0.1393,0.6175,-35.35,0.00,0.068970,0.166670,0.333330,0.533330,2.666666,3.666668,4.66667,4,30,10,0823~0824,sku_click_rate,top400以外
7,V4,0.1082,0.0866,0.1670,-59.80,0.00,0.102778,0.291665,0.406250,0.625003,0.715625,0.757812,0.80000,3,38,16,0823~0824,sku_click_rate,top400以外
8,V1,0.4407,3.7398,6.0821,4.52,1.90,4.350000,9.000000,14.700000,18.370000,27.800000,39.170000,64.00000,2825,886,620,0823~0824,avg点击位置,top400
9,V2,1.0000,3.5781,5.5137,0.00,1.80,4.300000,8.800000,13.515000,18.358000,26.430000,35.505000,52.10000,2787,902,642,0823~0824,avg点击位置,top400


In [21]:
# 导入odps_client库中的两个函数：get_odps_sql_result_as_df 用于执行SQL查询并将结果作为DataFrame返回, write_pandas_df_into_odps 用于将pandas DataFrame写入ODPS表
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

# 定义分区规范字符串，使用当前日期（年-月-日格式）作为分区值
partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

# 将DataFrame写入ODPS表
write_pandas_df_into_odps(
    df=all_user_variant_df,  # 要写入的DataFrame，这里是all_user_variant_df，包含了所有用户的变体信息
    table_name="summerfarm_ds.temp_search_ab_all_data_df",  # ODPS表名
    partition_spec=partition_spec,  # 分区规范
    overwrite=True,  # 如果表或分区已存在，是否覆盖
    lifecycle=30,  # 设置表的生命周期为30天
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

# 将开始日期格式化为字符串（年-月-日）
start_date_str = start_date.strftime("%Y-%m-%d")

# 定义SQL查询字符串，用于获取用户订单数据.
# 这段SQL的目的是：从订单表和用户分流表中，根据用户ID和日期进行关联，
# 统计每个用户在不同实验变体下的订单总金额、订单数量和平均订单金额。
order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

# 执行SQL查询并将结果作为DataFrame返回
user_orders_df = get_odps_sql_result_as_df(order_query)
# 显示DataFrame的前两行
user_orders_df.head(2)

2025-08-25 15:07:40 - INFO - DaraFrame字段合集:experiment_id,pt,type,ds,page_ame,create_time,uid,api_list,page_name,variant_list,search_times,api
2025-08-25 15:07:43 - INFO - Tunnel session created: <TableUploadSession id=202508251507436ddf321a151dbf24 project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20250825>
2025-08-25 15:07:46 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20250825, attemp:0
2025-08-25 15:07:56 - INFO - Tunnel session created: <InstanceDownloadSession id=202508251507556bdc321a151d4726 project_name=summerfarm_ds instance_id=20250825070746891gam0y6gwsod>
2025-08-25 15:07:56 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-08-23 00:00:00'
    AND     m_size = '

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250823,100041,V1,None,0,None
1,20250823,100111,V3,None,0,None


In [22]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count     8693.000000
mean       516.496983
std        757.736848
min          1.800000
25%        171.000000
50%        303.000000
75%        587.000000
max      25295.000000
Name: order_gmv, dtype: float64

In [23]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500      303.0000
0.750      587.0000
0.950     1551.4000
0.990     3331.2800
0.995     4407.5600
0.996     4820.8720
0.997     5030.3436
0.999     8344.2088
1.000    25295.0000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      52.000
0.05      84.982
0.25     171.000
0.50     302.000
0.75     584.850
0.95    1520.360
0.99    3167.520
Name: order_gmv, dtype: float64


In [24]:
user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(
    float
)

user_orders_below_6k_df["avg_order_gmv"].fillna(0.0, inplace=True)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df[
    "avg_order_gmv"
].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_48917/858576852.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_48917/858576852.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.3857,497.9858,581.8308,-3.05,296.0,595.0000,1005.98,1448.340,1914.6528,3296.6400,3772.860,5983.0,535086,1074,1074,0823~0824,order_gmv
1,V2,1.0000,513.6593,609.4625,0.00,317.0,586.0000,1076.00,1611.100,2024.0200,3250.4000,3871.576,6000.0,564768,1100,1100,0823~0824,order_gmv
2,V3,0.0457,478.6390,542.8957,-6.82,296.0,573.7225,1015.00,1431.550,1932.0200,2786.4900,3350.250,5886.4,510708,1067,1067,0823~0824,order_gmv
3,V4,0.4326,499.4640,589.2142,-2.76,298.9,579.1000,1063.36,1521.902,1872.2800,3248.8122,3998.490,5607.0,548162,1098,1098,0823~0824,order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.6674,397.9270,429.0401,-1.39,255.67,510.50,844.5,1058.200,1333.1256,2149.8320,2982.99,5386.0,427573,1074,1074,0823~0824,avg_order_gmv
1,V2,1.0000,403.5477,433.2589,0.00,252.92,510.93,864.0,1130.379,1463.4200,2141.2200,2674.92,6000.0,443701,1100,1100,0823~0824,avg_order_gmv
2,V3,0.3173,390.6662,414.5430,-3.19,246.00,500.00,819.7,1093.985,1352.5650,2090.3900,2559.43,4905.0,416841,1067,1067,0823~0824,avg_order_gmv
3,V4,0.7958,400.1775,429.7352,-0.84,250.00,510.00,806.6,1092.050,1523.8000,2121.9468,2557.20,5255.0,439195,1098,1098,0823~0824,avg_order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.3826,1.2755,0.7496,-1.55,1.0,1.0,2.0,2.0,3.0,4.0,5.00,15,1370,1074,1074,0823~0824,order_cnt
1,V2,1.0000,1.2956,0.7690,0.00,1.0,1.0,2.0,2.0,3.0,4.0,5.01,15,1424,1100,1100,0823~0824,order_cnt
2,V3,0.0647,1.2559,0.6424,-3.07,1.0,1.0,2.0,2.0,3.0,4.0,4.00,11,1340,1067,1067,0823~0824,order_cnt
3,V4,0.2384,1.2702,0.6562,-1.96,1.0,1.0,2.0,3.0,3.0,4.0,4.03,9,1394,1098,1098,0823~0824,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-0823~0824.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.3857,497.9858,581.8308,-3.05,296.00,595.0000,1005.98,1448.340,1914.6528,3296.6400,3772.860,5983.0,535086,1074,1074,0823~0824,order_gmv
1,V2,1.0000,513.6593,609.4625,0.00,317.00,586.0000,1076.00,1611.100,2024.0200,3250.4000,3871.576,6000.0,564768,1100,1100,0823~0824,order_gmv
2,V3,0.0457,478.6390,542.8957,-6.82,296.00,573.7225,1015.00,1431.550,1932.0200,2786.4900,3350.250,5886.4,510708,1067,1067,0823~0824,order_gmv
3,V4,0.4326,499.4640,589.2142,-2.76,298.90,579.1000,1063.36,1521.902,1872.2800,3248.8122,3998.490,5607.0,548162,1098,1098,0823~0824,order_gmv
4,V1,0.6674,397.9270,429.0401,-1.39,255.67,510.5000,844.50,1058.200,1333.1256,2149.8320,2982.990,5386.0,427573,1074,1074,0823~0824,avg_order_gmv
5,V2,1.0000,403.5477,433.2589,0.00,252.92,510.9300,864.00,1130.379,1463.4200,2141.2200,2674.920,6000.0,443701,1100,1100,0823~0824,avg_order_gmv
6,V3,0.3173,390.6662,414.5430,-3.19,246.00,500.0000,819.70,1093.985,1352.5650,2090.3900,2559.430,4905.0,416841,1067,1067,0823~0824,avg_order_gmv
7,V4,0.7958,400.1775,429.7352,-0.84,250.00,510.0000,806.60,1092.050,1523.8000,2121.9468,2557.200,5255.0,439195,1098,1098,0823~0824,avg_order_gmv
8,V1,0.3826,1.2755,0.7496,-1.55,1.00,1.0000,2.00,2.000,3.0000,4.0000,5.000,15.0,1370,1074,1074,0823~0824,order_cnt
9,V2,1.0000,1.2956,0.7690,0.00,1.00,1.0000,2.00,2.000,3.0000,4.0000,5.010,15.0,1424,1100,1100,0823~0824,order_cnt


In [25]:
all_p_values_df.to_csv(
    f"./data/搜索AB--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)